<a href="https://colab.research.google.com/github/mnedo/SMADIMO_GP5/blob/katya-residual-mlp/notebooks/katya_residual_mlp_oskelly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [348]:
import pandas as pd
import numpy as np

In [349]:
X = np.load('X (1).npy').astype(np.float32)
y = np.load('y (1).npy').astype(np.float32)

In [350]:
print(X.shape)
print(y.shape)

(9268, 588)
(9268,)


In [351]:
X_text = X[:,:384]
X_struct = X[:,384:]
print(X_text.shape)
print(X_struct.shape)
print(y.shape)

(9268, 384)
(9268, 204)
(9268,)


In [352]:
from sklearn.model_selection import train_test_split

train,temp = train_test_split(np.arange(len(X)),test_size=0.3,random_state=42)
val, test = train_test_split(temp,test_size=0.5,random_state=42)
print(len(train))
print(len(val))
print(len(test))

6487
1390
1391


In [353]:
X_train = X_struct[train]
Text_train = X_text[train]
y_train = y[train]

X_val = X_struct[val]
Text_val = X_text[val]
y_val = y[val]

X_test = X_struct[test]
Text_test = X_text[test]
y_test = y[test]


In [354]:
y_mean = y_train.mean()
y_std = y_train.std()

y_train_new = ((y_train - y_mean) / y_std)
y_val_new = ((y_val - y_mean) / y_std)
y_test_new = ((y_test - y_mean) / y_std)

print(y_mean)
print(y_std)

10.748254
0.79314494


In [355]:
print(X_train.shape, Text_train.shape, y_train.shape)
print(X_val.shape, Text_val.shape, y_val.shape)
print(X_test.shape, Text_test.shape, y_test.shape)

(6487, 204) (6487, 384) (6487,)
(1390, 204) (1390, 384) (1390,)
(1391, 204) (1391, 384) (1391,)


In [356]:
from sklearn.preprocessing import StandardScaler

scaler_text = StandardScaler()

Text_train_new = scaler_text.fit_transform(Text_train).astype(np.float32)
Text_val_new = scaler_text.transform(Text_val).astype(np.float32)
Text_test_new = scaler_text.transform(Text_test).astype(np.float32)

X_train_new = X_train.astype(np.float32)
X_val_new = X_val.astype(np.float32)
X_test_new = X_test.astype(np.float32)

In [357]:
import torch
import random

SEED =42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train_new)
X_val_t = torch.tensor(X_val_new)
X_test_t = torch.tensor(X_test_new)

Text_train_t = torch.tensor(Text_train_new)
Text_val_t = torch.tensor(Text_val_new)
Text_test_t = torch.tensor(Text_test_new)

y_train_t = torch.tensor(y_train_new).view(-1, 1)
y_val_t = torch.tensor(y_val_new).view(-1, 1)
y_test_t = torch.tensor(y_test_new).view(-1, 1)

print(X_train_t.dtype)
print(X_val_t.dtype)
print(X_test_t.dtype)
print(Text_train_t.dtype)
print(Text_val_t.dtype)
print(Text_test_t.dtype)
print(y_train_t.dtype)
print(y_val_t.dtype)
print(y_test_t.dtype)

torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32


Для обучения модели данные были переведены в формат torch.tensor, так как PyTorch работает с тензорами. Все признаки и таргет были приведены к типу torch.float32. Это считается стандартным типом для обучения нейросетей в PyTorch. Он занимает меньше памяти чем float64, и совпадает с типом весов модели по умолчанию.

In [358]:
train_dataset = TensorDataset(X_train_t, Text_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, Text_val_t, y_val_t)
test_dataset = TensorDataset(X_test_t, Text_test_t, y_test_t)


In [359]:
!pip install comet_ml

from comet_ml import Experiment
from getpass import getpass

In [360]:
my_api = getpass('API key:')

config = {'architecture': 'two_branch_residual_mlp','epochs': 250,
'batch_size': 64,'lr': 0.0002,'weight_decay': 1e-4,'branch_dim': 256,
'hidden_dim':512,'dropout': 0.15, 'patience': 35,
'optimizer': 'AdamW','loss': 'MSELoss','seed': 42}

experiment = Experiment(api_key = my_api, project_name = 'oskelly-mlp', workspace= 'gp5_team1')
experiment.set_name('mlp_run_12 Role-C TwoBranchResidualMLP dropout = 0.15 lr = 0.0002')
experiment.log_parameters(config)

API key:··········


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-mlp/999fa91e66b6478c808912a0b6518169



In [361]:
g = torch.Generator()
g.manual_seed(config['seed'])

train_loader = DataLoader(train_dataset,batch_size = config['batch_size'],shuffle=True, generator=g)
val_loader = DataLoader(val_dataset,batch_size=config['batch_size'])
test_loader = DataLoader(test_dataset,batch_size = config['batch_size'])

In [362]:
import torch.nn as nn
class ResidualBlock(nn.Module):
  def __init__(self, hidden_dim, dropout):
    super().__init__()
    self.block = nn.Sequential(nn.Linear(hidden_dim, hidden_dim),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden_dim, hidden_dim))
    self.relu = nn.ReLU()
  def forward(self, x):
    out = self.block(x) + x
    return self.relu(out)

In [363]:
class TwoBranchMLP( nn.Module ):
  def __init__(self, struct_dim, text_dim, branch_dim, hidden_dim, dropout):
    super().__init__()
    self.tabular_encoder = nn.Sequential(nn.Linear(struct_dim, branch_dim),nn.BatchNorm1d(branch_dim),nn.ReLU(),
nn.Dropout(dropout),nn.Linear(branch_dim, branch_dim),nn.ReLU())

    self.text_encoder = nn.Sequential(nn.Linear(text_dim, branch_dim),nn.BatchNorm1d(branch_dim),nn.ReLU(),
nn.Dropout(dropout), nn.Linear(branch_dim, branch_dim),nn.ReLU())

    self.merger = nn.Sequential(nn.Linear(branch_dim * 2, hidden_dim),
                                nn.ReLU(),nn.Dropout(dropout))
    self.refiner = ResidualBlock(hidden_dim, dropout)
    self.output = nn.Linear(hidden_dim, 1)

  def forward(self, x_struct, x_text):
    struct_out = self.tabular_encoder(x_struct)
    text_out = self.text_encoder(x_text)
    merged = torch.cat([struct_out, text_out], dim=1)
    x = self.merger(merged)
    x = self.refiner(x)

    return self.output(x)

In [364]:
print(X_train_t.shape)
print(Text_train_t.shape)

torch.manual_seed(config ['seed'])
torch.cuda.manual_seed_all(config ['seed'])

model = TwoBranchMLP(struct_dim = X_train_t.shape[1],
text_dim = Text_train_t.shape[1],branch_dim = config['branch_dim'],
hidden_dim = config['hidden_dim'],dropout = config['dropout'])

torch.Size([6487, 204])
torch.Size([6487, 384])


In [365]:
optimizer = torch.optim.AdamW(model.parameters(),lr = config['lr'],weight_decay = config['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience = 10)
criterion = nn.MSELoss()

In [366]:
import copy

train_losses = []
val_losses = []

best_model_1 = None
best_epoch = 0
best_val = float('inf')
no_improve = 0

for epoch in range (config['epochs']):
  model.train()
  train_loss = 0
  for x_batch, text_batch, y_batch in train_loader:
    optimizer.zero_grad()
    output = model(x_batch, text_batch)
    loss = criterion(output, y_batch)

    loss.backward()
    optimizer.step()

    train_loss = train_loss + float(loss.detach())

  train_loss = train_loss / len(train_loader)
  train_losses.append(train_loss)
  model.eval()
  val_loss = 0
  with torch.no_grad():
    for x_batch, text_batch, y_batch in val_loader:
      output = model(x_batch, text_batch)
      loss = criterion(output, y_batch)
      val_loss = val_loss + float(loss.detach())

  val_loss = val_loss /len(val_loader)
  val_losses.append(val_loss)

  scheduler.step(val_loss)
  experiment.log_metric('train_loss', train_loss, step = epoch + 1)
  experiment.log_metric('val_loss', val_loss, step = epoch + 1)


  if val_loss < best_val:
    print('нашли лучшую модель',round(val_loss, 4))
    best_val = val_loss
    best_model_1 = copy.deepcopy(model.state_dict())
    best_epoch = epoch + 1
    no_improve = 0
  else:
    no_improve += 1
    if no_improve >= config['patience']:
      print('Ранняя остановка на эпохе:', epoch + 1)
      break



  print('Эпоха:',epoch + 1)
  print('ошибка на обучении:',round(train_loss, 4))
  print('ошибка на валидации:',round(val_loss, 4))

model.load_state_dict(best_model_1)
print('')
print('Лучшая эпоха:', best_epoch)
print('Лучший ошибка на валидации:', round(best_val, 4))

experiment.log_metric('best_epoch', best_epoch)
experiment.log_metric('best_val_loss', round(best_val, 4))

нашли лучшую модель 0.4752
Эпоха: 1
ошибка на обучении: 0.6433
ошибка на валидации: 0.4752
нашли лучшую модель 0.4395
Эпоха: 2
ошибка на обучении: 0.3955
ошибка на валидации: 0.4395
нашли лучшую модель 0.403
Эпоха: 3
ошибка на обучении: 0.3328
ошибка на валидации: 0.403
нашли лучшую модель 0.3915
Эпоха: 4
ошибка на обучении: 0.2895
ошибка на валидации: 0.3915
нашли лучшую модель 0.3894
Эпоха: 5
ошибка на обучении: 0.2693
ошибка на валидации: 0.3894
нашли лучшую модель 0.3812
Эпоха: 6
ошибка на обучении: 0.2375
ошибка на валидации: 0.3812
нашли лучшую модель 0.3719
Эпоха: 7
ошибка на обучении: 0.2106
ошибка на валидации: 0.3719
Эпоха: 8
ошибка на обучении: 0.1926
ошибка на валидации: 0.377
Эпоха: 9
ошибка на обучении: 0.1742
ошибка на валидации: 0.3835
нашли лучшую модель 0.3628
Эпоха: 10
ошибка на обучении: 0.1564
ошибка на валидации: 0.3628
Эпоха: 11
ошибка на обучении: 0.145
ошибка на валидации: 0.3663
нашли лучшую модель 0.36
Эпоха: 12
ошибка на обучении: 0.1366
ошибка на валидации:

In [367]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
model.load_state_dict(best_model_1)
model.eval()
test_loss = 0

preds = []
targets = []
print('посчитаем метрики на тесте')
with torch.no_grad():
  for x_batch, text_batch, y_batch in test_loader:
    output = model(x_batch, text_batch)
    loss = nn.MSELoss()(output, y_batch)

    test_loss = test_loss + float(loss.detach())
    preds.append(output.detach())
    targets.append(y_batch.detach())

pred_scaled = torch.cat(preds)
pred_scaled = pred_scaled.cpu().numpy().ravel()

true_scaled = torch.cat(targets)
true_scaled = true_scaled.cpu().numpy().ravel()

pred_log = pred_scaled * y_std + y_mean
true_log = true_scaled * y_std + y_mean

pred_price = np.expm1(pred_log)
true_price = np.expm1(true_log)

pred_price = np.maximum(pred_price, 0)
mse = mean_squared_error(true_price, pred_price)
diff = true_price - pred_price
abs_diff = np.abs(diff / true_price)

print('Ошибка на тесте (test_loss):',
      round(test_loss / len(test_loader), 4))
print('Средняя абсолютная ошибка (mae) :',
      round(mean_absolute_error(true_price, pred_price), 2))
print('Среднеквадратичная ошибка (rmse):',
      round(np.sqrt(mse), 2))
print('Средняя процентная ошибка (mape):',
      round(np.mean(abs_diff) *100, 2),'%')
print('Коэфф детерминации (r2):',
      round(r2_score(true_price, pred_price), 4))


experiment.log_metric('test_loss', round(test_loss / len(test_loader), 4))
experiment.log_metric('test_mae', round(mean_absolute_error(true_price, pred_price), 2))
experiment.log_metric('test_rmse', round(np.sqrt(mse), 2))
experiment.log_metric('test_mape', round(np.mean(abs_diff) *100, 2))
experiment.log_metric('test_r2', round(r2_score(true_price, pred_price), 4))

torch.save(best_model_1, 'best_model.pt')
experiment.log_model('best_model', 'best_model.pt')
experiment.end()

посчитаем метрики на тесте
Ошибка на тесте (test_loss): 0.3062
Средняя абсолютная ошибка (mae) : 19597.58
Среднеквадратичная ошибка (rmse): 34288.72
Средняя процентная ошибка (mape): 34.45 %
Коэфф детерминации (r2): 0.6248


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : mlp_run_12 Role-C TwoBranchResidualMLP dropout = 0.15 lr = 0.0002
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-mlp/999fa91e66b6478c808912a0b6518169
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     best_epoch      : 57
COMET INFO:     best_val_loss   : 0.3408
COMET INFO:     test_loss       : 0.3062
COMET INFO:     test_mae        : 19597.58
COMET INFO:     test_mape       : 34.45000076293945
COMET INFO:     test_r2         : 0.6248
COMET INFO:     test_rmse       : 34288.72
COMET INFO:     train_loss [92] : (0.022938425743988917, 0.6432635567936242)
COMET INFO:     val_loss [92]   : (0.34080844711173663, 0.